In [0]:
# step 1--> read customers table from bronze layer and store in df
# step 2---> dispaly and identify the cleaning to be done

#step3---> apply required transformation steps which u identified and store in df 
#step4__> final df  we will apply scd  type 1 and store in silver schema  before silver schema using sql
#step5---> write a function in a same notebook clean and tranform the customer data from beonze and return df


#step 6---> write a function which can do scd1-if we provide teh parameters like source dataframe, business_key and target_table [after ctreating this function scd1 we will take it  to common_utils and reuse it here and in other notebooks


#step7--> Fully modularize the code ( hardcoded values to json config, proper logging setup and all the resusabl

In [0]:
import pyspark.sql.functions  as F
from pyspark.sql.functions import lower, col
from pyspark.sql.functions import expr, from_unixtime
df = spark.read.table("retaildataplatform.bronze.sqlserver_customers")



def transform_customer(df):

    df = df.withColumn("customer_name", F.lower(df["customer_name"]))\
        .withColumn("name_parts", F.split(F.col("customer_name"),","))\
        .withColumn("first_name", F.expr("get(name_parts,1)"))\
        .withColumn("last_name", F.expr("get(name_parts,0)"))\
        .withColumn("city", F.upper(df["city"]))\
        .withColumn("postcode", F.regexp_replace(F.col("postcode"), "\.0$", ""))\
        .withColumn("valid_from", F.from_unixtime(F.expr("try_cast(valid_from as bigint)"))) \
        .withColumn("valid_to", F.from_unixtime(F.expr("try_cast(valid_to as bigint)")))\
        .withColumn('country',F.lit('USA'))

    df = df.drop("name_parts","customer_name","file_path")

    df = df.dropDuplicates(["customer_id"])

    return df.select(
        "customer_id",
        "first_name",
        "last_name",
        "tax_id",
        "tax_code",
        "country",
        "state",
        "city",
        "postcode",
        "street",
        "number",
        "unit",
        "region",
        "district",
        "lon",
        "lat",
        "ship_to_address",
        "valid_from",
        "valid_to",
        "units_purchased",
        "loyalty_segment",
        "last_update_ts"

    )

df = spark.read.table("retaildataplatform.bronze.sqlserver_customers")
df = df.withColumnRenamed("last_upadte_ts", "last_update_ts")
cleaned_customer = transform_customer(df)


if spark.catalog.tableExists("retaildataplatform.silver.customers"):
    print("Table Exists - Now Proceeding with SCD 1")

    #Load target

    target = DeltaTable.forname(spark,target_table)

    #SCD Type 1

    #Match in source and target-->Key exists in target

    (
        target
        .merge(
            cleaned_customer.alias("source"),
            "target.customer_id = source.customer_id"
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )


 



else:
    print("Table Does Not Exist - Now Creating Table")
    spark.sql("""Create Schema if not exists silver """)
    cleaned_customer.write.format("delta").mode("overwrite").saveAsTable("retaildataplatform.silver.customers")

     


     


In [0]:
dbutils.widgets.text("path", "")
config_path = dbutils.widgets.get("path")


In [0]:


dbutils.widgets.text("path", "/Workspace/Users/shashankkotagiri22@gmail.com/retaildataplatform/src/silver/config/config_customers.json")
config_path = dbutils.widgets.get("path")

     

import json

import pyspark.sql.functions as F
from pyspark.sql.window import Window

from common_utils.logging import get_logger
from common_utils.generic_functions import rename_columns, safe_cast
from common_utils.scdfunctions import scd_type_1


# ============================================================
# 0. LOGGER
# ============================================================

logger = get_logger("customers_silver")


# ============================================================
# 1. READ CONFIG
# ============================================================

dbutils.widgets.text("path", "/Workspace/Users/shashankkotagiri22@gmail.com/retaildataplatform/src/silver/config/config_customers.json")
config_path = dbutils.widgets.get("path")

if not config_path.startswith("/"):
    config_path = "/Workspace/Users/shashankkotagiri22@gmail.com/retaildataplatform/src/silver/config/config_customers.json"

logger.info("Reading Silver configuration from %s", config_path)

with open(config_path, "r") as f:
    config = json.load(f)


# ============================================================
# 2. SOURCE / TARGET
# ============================================================

source_table = config["source"]["source_table"]
source_name = config["source"]["source_name"]
target_table = config["target"]["target_table"]


# ============================================================
# 3. CUSTOMER TRANSFORMATION
# ============================================================

def transform_customer(df, config):

    logger.info("[%s] Starting customer transformation", source_name)

    # --------------------------------------------------------
    # Rename columns
    # --------------------------------------------------------

    df = rename_columns(
        df,
        config.get("rename_columns", {})
    )


    # --------------------------------------------------------
    # Customer name
    # --------------------------------------------------------

    if "customer_name" in df.columns:

        logger.info("[%s] Transforming customer name", source_name)

        df = (
            df
            .withColumn("customer_name", F.lower(F.col("customer_name")))
            .withColumn("name_parts", F.split(F.col("customer_name"), ","))
            .withColumn("first_name", F.trim(F.expr("get(name_parts, 1)")))
            .withColumn("last_name", F.trim(F.expr("get(name_parts, 0)")))
        )


    # --------------------------------------------------------
    # City
    # --------------------------------------------------------

    if "city" in df.columns:

        logger.info("[%s] Standardising city", source_name)

        df = df.withColumn("city", F.upper(F.col("city")))


    # --------------------------------------------------------
    # Postcode
    # --------------------------------------------------------

    if "postcode" in df.columns:

        logger.info("[%s] Cleaning postcode", source_name)

        df = df.withColumn("postcode", F.regexp_replace(F.col("postcode"), r"\.0$", "")
        )


    # --------------------------------------------------------
    # Country
    # --------------------------------------------------------

    logger.info("[%s] Adding country", source_name)

    df = df.withColumn("country", F.lit("USA"))


    # --------------------------------------------------------
    # Drop columns
    # --------------------------------------------------------

    drop_columns = [
        c for c in config.get("drop_columns", [])
        if c in df.columns
    ]

    if drop_columns:

        logger.info(
            "[%s] Dropping columns: %s",
            source_name,
            drop_columns
        )

        df = df.drop(*drop_columns)


    # --------------------------------------------------------
    # Cast columns
    # --------------------------------------------------------

    df = safe_cast( df, config.get("cast_columns", {}) )


    # --------------------------------------------------------
    # Deduplication
    # --------------------------------------------------------

    dedup = config.get("deduplication", {})

    if dedup.get("enabled", False):

        logger.info("[%s] Removing duplicate records", source_name)

        key_columns = dedup["key_columns"]

        order_expressions = []

        for item in dedup["order_by"]:

            col = F.col(item["column"])

            if item.get("direction", "asc").lower() == "desc":

                col = (
                    col.desc_nulls_last()
                    if item.get("nulls", "last").lower() == "last"
                    else col.desc_nulls_first()
                )

            else:

                col = (
                    col.asc_nulls_last()
                    if item.get("nulls", "last").lower() == "last"
                    else col.asc_nulls_first()
                )

            order_expressions.append(col)

        window = (
            Window
            .partitionBy(*key_columns)
            .orderBy(*order_expressions)
        )

        df = (
            df
            .withColumn("rn", F.row_number().over(window))
            .filter(F.col("rn") == 1)
            .drop("rn")
        )


    # --------------------------------------------------------
    # Final columns
    # --------------------------------------------------------

    logger.info("[%s] Selecting final Silver columns", source_name)

    df = df.select(
        *config.get("select_columns", [])
    )

    logger.info("[%s] Customer transformation completed", source_name)

    return df


# ============================================================
# 4. READ BRONZE
# ============================================================

logger.info(
    "[%s] Reading from Bronze table %s",
    source_name,
    source_table
)

df = spark.read.table(source_table)

logger.info(
    "[%s] Successfully read Bronze table",
    source_name
)


# ============================================================
# 5. TRANSFORM CUSTOMER
# ============================================================

cleaned_customer = transform_customer(
    df,
    config
)


# ============================================================
# 6. SCD / INITIAL LOAD
# ============================================================

if spark.catalog.tableExists(target_table):

    scd = config.get("scd", {})

    if scd.get("type") == 1:

        logger.info(
            "[%s] Starting SCD Type 1",
            source_name
        )

        scd_type_1(
            spark=spark,
            source_df=cleaned_customer,
            target_table=target_table,
            key_columns=scd.get("key_columns", [])
        )

    else:

        raise ValueError(
            f"Unsupported SCD type [{scd.get('type')}] "
            f"for source [{source_name}]"
        )

else:

    logger.info(
        "[%s] Target table does not exist - creating %s",
        source_name,
        target_table
    )

    spark.sql("""
        CREATE SCHEMA IF NOT EXISTS
        retaildataplatform.silver
    """)

    (
        cleaned_customer
        .write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(target_table)
    )

    logger.info(
        "[%s] Silver table created successfully",
        source_name
    )


# ============================================================
# 7. COMPLETION
# ============================================================

logger.info(
    "[%s] Bronze to Silver pipeline completed successfully",
    source_name
)
     